# 1 · What does the model actually train on?

Chapter 6 / Day 8. Follow a real (tiny, authored) text corpus into the Chapter 5 decoder. The objective is to specify **which predictions count**, not to train a useful language model.

By the end: explain document splits, byte tokens, shifted windows, shuffling, token budgets, and correctly weighted accumulation. No download or GPU is needed. Read [Chapter 6](../../book/chapters/06-pretraining-as-a-controlled-system.md) alongside this lab.

Prediction first: could two runs process the same number of tensor positions but learn from different numbers of targets? Write your explanation before running the references.


In [ ]:
from pathlib import Path
import sys, tempfile
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import torch
import matplotlib.pyplot as plt
from dongxi_llms import pretraining_lab as lab
from dongxi_llms import pretraining_visuals as viz
from dongxi_llms.decoder_lab import parameter_count
torch.set_num_threads(1)
print("CPU teaching lab", torch.__version__)
# Source notebooks stay unexecuted; all checkpoints below use temporary directories.


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Training system with data, windows, and token loss highlighted](../figures/chapter-06/01-system.png)


In [ ]:
fig = viz.process_map([0, 1, 3]); plt.show()


## 1. Split documents before making examples

These ten original sentences are a **teaching fixture**, not a representative language dataset. Eight documents train the model; two are held out. We detect normalized exact overlap, not paraphrases or near duplicates.

**Exercise.** Predict what happens if a validation sentence is copied into training with different capitalization and extra spaces. Why would splitting overlapping windows first be dangerous?

**Reference solution.** Hold out source documents first. A copied passage can let a model recognize an evaluation example without generalizing. A document-level split reduces one route to leakage; it cannot detect all duplication.


In [ ]:
lab.check_splits(lab.TRAIN, lab.VALID)
print("Training documents:", len(lab.TRAIN), "validation documents:", len(lab.VALID))
try:
    lab.check_splits((("train", "The CAT"),), (("valid", " the  cat "),))
except ValueError as error:
    print("Expected rejection:", error)
else:
    raise AssertionError("Contamination was not detected")


## 2. From text to token IDs, then next-token labels

Our transparent tokenizer maps every UTF-8 byte to its ID (0–255), plus EOS=256 and BOS=257. Vocabulary size is 258. This is **not BPE** and not a recommendation for production tokenization. It reconnects Chapter 2's byte fallback to training accounting.

For a document, form `[BOS] + bytes + [EOS]`, then `inputs = sequence[:-1]` and `labels = sequence[1:]`. Shift once; our loss helper does not shift again.

**Exercise.** Will “数” contribute one target? Where does EOS enter the target count?

**Reference solution.** “数” occupies three UTF-8 bytes, so this document contributes three byte targets plus EOS. BOS is context, not a predicted target in this fixture.


In [ ]:
print("数 bytes:", list("数".encode("utf-8")))
mini = lab.make_windows((("example", "abc"),), length=3)
print("Input IDs:", mini["x"].tolist())
print("Target IDs:", mini["y"].tolist())
assert mini["y"].tolist() == [[97, 98, 99], [lab.EOS, -100, -100]]
train, valid = lab.fixture()
print("Window shapes:", train["x"].shape, train["y"].shape)


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Input and target ID grids showing a document tail and ignored labels](../figures/chapter-06/01-windows.png)


In [ ]:
fig = viz.window_map(train, indices=(0, 1, 2)); plt.show()


Read each row left to right. Inputs and targets refer to adjacent tokens in the source. A tail uses EOS as the input padding value, but target -100 means **no loss at that position**. EOS can also be a real target; do not discard every EOS ID.

Windows never cross documents. They restart context and RoPE positions; this is a deliberate simple policy, not dense packing. Right padding cannot influence valid earlier positions through causal attention. A loss mask would not, by itself, prevent arbitrary padding or another document from being attended to.

**Perturbation.** Change `length` from 16 to 8 below. Predict whether the total valid target count changes, and whether the context available to a target changes.

**Reference solution.** Every target appears once under this policy, but smaller windows give some targets less context and may change padding overhead. Same target count does not imply the same conditional learning problem.


In [ ]:
for length in (8, 16, 32):
    windows = lab.make_windows(lab.TRAIN, length)
    valid_targets = int((windows["y"] != lab.IGNORE).sum())
    expected = sum(len(text.encode("utf-8")) + 1 for _, text in lab.TRAIN)
    assert valid_targets == expected
    print(dict(length=length, windows=len(windows["x"]), valid_targets=valid_targets,
               processed_positions=windows["x"].numel(),
               useful_fraction=valid_targets/windows["x"].numel()))


## 3. Shuffling changes the gradient sequence

**Exercise.** If each window occurs once per epoch, why could its order still matter?

**Reference solution.** The second window is evaluated at parameters already changed by the first. AdamW also carries gradient history. Equal membership does not imply equal trajectories. Our single-process stream saves its permutation, cursor, epoch, and generator state; a production loader has more state.


In [ ]:
stream = lab.WindowStream(len(train["x"]))
first_epoch = stream.take(len(train["x"]))
assert sorted(first_epoch) == list(range(len(train["x"])))
print("First shuffled document IDs:", [train["documents"][i] for i in first_epoch[:8]])
saved = stream.state_dict()
expected_next = stream.take(5)
replayed = lab.WindowStream(len(train["x"]))
replayed.load_state_dict(saved)
assert replayed.take(5) == expected_next
print("Next epoch can be replayed:", expected_next)


## 4. A batch budget is not automatically a target budget

For S updates, microbatch size b, length T, A accumulation steps and R data-parallel ranks, **full valid windows** give N = S × b × T × A × R. Padding or ignored prompt labels make this an upper bound on supervised targets, not an exact count. Repeated epochs increase presentations, not unique data.

**Exercise.** Predict the maximum targets in our 24-update recipe (b=1, T=16, A=2, R=1). Explain why the runtime must still count labels.

**Reference solution.** 768 positions is the ceiling. The true denominator is the number of labels other than -100; log that counter separately from optimizer steps and unique corpus tokens.


In [ ]:
S, b, T, A, R = 24, 1, 16, 2, 1
print("Position budget:", S*b*T*A*R)
print("Unique fixture target positions:", int((train["y"] != lab.IGNORE).sum()))
print("Validation target positions:", int((valid["y"] != lab.IGNORE).sum()))


## 5. Accumulation must preserve the objective

**Deep question.** Should a microbatch with one valid target have as much influence as one with sixteen?

For microbatch j, use its **summed** NLL divided by N, the total valid targets across the update. Backward each contribution; step once after accumulation. Averaging microbatch means gives each microbatch equal weight, which is a different objective when valid lengths differ.

**Reference implementation.** Here is the key loop, exposed rather than hidden inside a training framework.


In [ ]:
model = lab.make_model(dtype=torch.float64)
audit = lab.accumulation_audit()
x, y = audit["x"], audit["y"]
N = int((y != lab.IGNORE).sum())
model.zero_grad(set_to_none=True)
for j in range(len(x)):
    contribution = lab.loss_sum(model, x[j:j+1], y[j:j+1]) / N
    contribution.backward()
# Do NOT zero gradients or optimizer.step() inside this loop.
assert audit["correct_error"] < 1e-10
assert audit["wrong_error"] > 1e-4
print({k: v for k, v in audit.items() if k not in ("x", "y")})


Saved reference preview—not a live result. Run the following cell to regenerate from this session.

![Valid-token weighting and measured gradient mismatch](../figures/chapter-06/01-accumulation.png)


In [ ]:
fig = viz.accumulation_plot(audit); plt.show()


**Explanation.** The correct gradient agrees to numerical tolerance; the deliberately wrong denominator disagrees. This comparison holds parameters fixed and uses a deterministic, dropout-free decoder. Different floating-point reductions, dropout masks or cross-example operations can prevent bitwise agreement in other setups. Distributed gradient averaging introduces another normalization factor that this single-rank lab does not test.

## Discussion and evidence boundary

Could a recipe look faster simply because it predicts fewer real targets? Explain what you would log: processed positions/s, valid targets/s, total target presentations, padding fraction, and data identity. Neither throughput figure alone measures model quality.

You have audited a data-and-objective contract, not approved a production corpus or demonstrated mastery by executing cells. Next: [AdamW, schedules, and stability](02_adamw_schedule_and_stability.ipynb).
